# 03 - Fine-tune BanglaBERT

Fine-tune `sagorsarker/bangla-bert-base` for Bangla toxic comment classification.

**Run this notebook on Kaggle or Colab with GPU enabled.**

In [1]:
# Google Colab setup (does nothing when run locally).
# Clones the repo into Google Drive so results/ survive disconnects, then installs dependencies.
# Before the first run: Runtime > Change runtime type > T4 GPU, and upload train.csv (see below).
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Set_A'
    if not os.path.isdir(REPO_DIR):
        !git clone -b bangal_profanity https://github.com/mehediparvez/Set_A.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}/BanglaProfanityDetection/notebooks
    !pip install -q transformers datasets accelerate scikit-learn better-profanity groq google-generativeai pyyaml seaborn tabulate
    if not os.path.exists('../data/train.csv'):
        print(f'Missing dataset: upload train.csv to {REPO_DIR}/BanglaProfanityDetection/data/train.csv')

Mounted at /content/drive
Already up to date.
/content/drive/MyDrive/Set_A/BanglaProfanityDetection/notebooks
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.16.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboar

In [2]:
import sys
import time
import torch
sys.path.append('..')

from src.dataset import load_dataset
from src.train import train_with_multiple_runs, train_learning_curve
from src.evaluate import evaluate_model, get_classification_report, aggregate_runs
from src.model import load_trained_model
from src.results import build_evaluation, save_evaluation
from src.utils import load_config, save_metrics, ensure_dirs

config = load_config('../configs/config.yaml')
model_config = config['models']['banglabert']
MODEL_NAME = model_config['name']
RESULTS_KEY = 'banglabert'  # results/<key>/ is read by notebook 06 and main.py

ensure_dirs('../results', '../plots', '../saved_models')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Model: {MODEL_NAME}')

Device: cuda
Model: sagorsarker/bangla-bert-base


## Load Dataset

In [3]:
DATA_PATH = '../' + config['data']['train_path']

train_texts, test_texts, train_labels, test_labels = load_dataset(
    DATA_PATH,
    text_column=config['data']['text_column'],
    label_column=config['data']['label_column'],
    test_size=config['data']['test_size'],
    random_seed=config['data']['random_seed'],
)
print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')

Train: 12730, Test: 3183


## Train (3 Runs with Error Bars)

In [4]:
all_metrics = train_with_multiple_runs(
    MODEL_NAME,
    train_texts, test_texts,
    train_labels, test_labels,
    model_config,
    output_dir='../results',
    num_runs=config['training']['num_runs'],
    seed=config['data']['random_seed'],
    val_size=config['training']['val_size'],
)


Run 1/3


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  660MB            

model.safetensors: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.045377,0.245447,4,0.932139,0.932059,0.932406,0.932139


Run 3 - Accuracy: 0.9321, F1: 0.9321


In [5]:
summary = aggregate_runs(all_metrics)
print('\n=== BanglaBERT Results (3 runs) ===')
for key, stats in summary.items():
    if 'eval_' in key:
        print(f"  {key}: {stats['mean']:.4f} +/- {stats['std']:.4f}")

save_metrics(summary, f'../results/{RESULTS_KEY}/summary.json')
save_metrics(all_metrics, f'../results/{RESULTS_KEY}/all_runs.json')


=== BanglaBERT Results (3 runs) ===
  eval_loss: 0.4020 +/- 0.1120
  eval_accuracy: 0.9270 +/- 0.0037
  eval_f1: 0.9270 +/- 0.0036
  eval_precision: 0.9271 +/- 0.0038
  eval_recall: 0.9270 +/- 0.0037


## Learning Curve (25%, 50%, 100%)

In [ ]:
lc_results = train_learning_curve(
    MODEL_NAME,
    train_texts, test_texts,
    train_labels, test_labels,
    model_config,
    fractions=[0.25, 0.5, 1.0],
    output_dir='../results',
    seed=config['data']['random_seed'],
    val_size=config['training']['val_size'],
)

print('\n=== Learning Curve ===')
for r in lc_results:
    print(f"  {r['fraction']*100:.0f}% ({r['num_samples']} samples): "
          f"Acc={r['eval_accuracy']:.4f}, F1={r['eval_f1']:.4f}")

save_metrics(lc_results, f'../results/{RESULTS_KEY}/learning_curve.json')


Training on 25% of data (3182 samples)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.299344,0.306744,0.893417,0.891992,0.901246,0.893417
2,0.215179,0.259797,0.909091,0.908570,0.911154,0.909091
3,0.107066,0.357877,0.921630,0.921348,0.922565,0.921630
4,0.058764,0.367303,0.905956,0.905663,0.906531,0.905956
5,0.039611,0.428255,0.902821,0.902564,0.903190,0.902821


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.275641,0.237774,0.912804,0.912726,0.912904,0.912804
2,0.215294,0.270058,0.907306,0.907315,0.907328,0.907306
3,0.097950,0.315607,0.922231,0.922077,0.922890,0.922231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.275641,0.237774,0.912804,0.912726,0.912904,0.912804
2,0.215294,0.270058,0.907306,0.907315,0.907328,0.907306
3,0.097950,0.315607,0.922231,0.922077,0.922890,0.922231


## Evaluate Best Model

In [ ]:
best_model_dir = f'../results/{MODEL_NAME.split("/")[-1]}/run_1/best_model'
model, tokenizer = load_trained_model(best_model_dir)
if torch.cuda.is_available():
    model = model.cuda()

start_time = time.time()
_, preds = evaluate_model(model, tokenizer, test_texts, test_labels, max_length=model_config['max_length'])
evaluation = build_evaluation(test_labels, preds, time.time() - start_time)
save_evaluation(evaluation, '../results', RESULTS_KEY)

print('\n=== Best Model Evaluation ===')
for k, v in evaluation['metrics'].items():
    print(f'  {k}: {v:.4f}')

print('\n=== Classification Report ===')
print(get_classification_report(test_labels, preds))

print('\n=== Confusion Matrix ===')
print(evaluation['confusion_matrix'])

In [ ]:
# Colab: make sure everything written to Google Drive is actually uploaded before the runtime ends
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All results saved to Google Drive.')